# Crop Leaf Disease Detection - Demo App (no training needed)
Loads the already-trained model and launches the Gradio app.

**Before running:** download these two files from the training notebook (section 10) and keep them ready:
`crop_disease_model.keras` and `class_names.json`.

Runtime -> Change runtime type -> T4 GPU is optional (the app also runs on CPU, just a bit slower).

In [1]:
!pip install -q gradio

## 1. Upload the model files
A file picker will appear. Select both `crop_disease_model.keras` and `class_names.json`.

In [2]:
import os
from google.colab import files

if not (os.path.exists("crop_disease_model.keras") and os.path.exists("class_names.json")):
    files.upload()

print("crop_disease_model.keras:", os.path.exists("crop_disease_model.keras"))
print("class_names.json        :", os.path.exists("class_names.json"))

Saving crop_disease_model.keras to crop_disease_model (1).keras
crop_disease_model.keras: True
class_names.json        : True


## 2. Load the model

In [3]:
import json
import tensorflow as tf

model = tf.keras.models.load_model("crop_disease_model.keras")
class_names = json.load(open("class_names.json"))
print("Model loaded. Classes:", len(class_names))

Model loaded. Classes: 44


## 3. Severity estimation, treatment advice and the app

In [4]:
import cv2
import numpy as np

def estimate_severity(img_np):
    """Estimate the percentage of diseased leaf area using HSV colour segmentation."""
    img = cv2.resize(np.array(img_np).astype(np.uint8), (448, 448))
    h, s, v = cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2HSV))

    # 1) Separate the leaf from the background (leaf pixels are more saturated)
    fg = (((s > 35) & (v > 30)).astype(np.uint8)) * 255
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None, img
    leaf = np.zeros_like(fg)
    cv2.drawContours(leaf, [max(cnts, key=cv2.contourArea)], -1, 255, -1)
    if cv2.countNonZero(leaf) < 0.05 * leaf.size:
        return None, img
    leaf = cv2.erode(leaf, np.ones((5, 5), np.uint8))  # ignore the leaf border

    # 2) Inside the leaf, anything that is not green is treated as diseased
    green = (((h >= 30) & (h <= 90) & (s > 40) & (v > 40)).astype(np.uint8)) * 255
    dis = cv2.bitwise_and(leaf, cv2.bitwise_not(green))
    dis = cv2.morphologyEx(dis, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    pct = 100.0 * cv2.countNonZero(dis) / max(cv2.countNonZero(leaf), 1)
    red = img.copy()
    red[dis > 0] = (255, 0, 0)
    return pct, cv2.addWeighted(img, 0.5, red, 0.5, 0)

def severity_level(pct):
    if pct < 10:
        return "Mild"
    if pct < 30:
        return "Moderate"
    return "Severe"

In [5]:
# (keywords, disease type, recommended active ingredients, organic / cultural measures)
# Order matters: the first matching entry is used.
MEDICINE = [
    (("mite",), "Pest (mite)",
     "Abamectin or Spiromesifen (as per product label).",
     "Neem oil spray, strong water spray on the underside of leaves."),
    (("virus", "curl", "mosaic", "yellow", "greening"), "Viral / vector-borne disease",
     "There is no direct cure for viruses. Control the insect vectors (aphid, whitefly, psyllid) with Imidacloprid or Thiamethoxam.",
     "Neem oil (5 ml/litre), remove and destroy infected plants, use disease-free seed or seedlings."),
    (("bacterial",), "Bacterial disease",
     "Copper oxychloride, together with Streptocycline (as per product label).",
     "Remove infected leaves, avoid overhead irrigation."),
    (("late_blight",), "Fungal disease (late blight)",
     "Metalaxyl + Mancozeb, or Cymoxanil + Mancozeb.",
     "Burn infected leaves, follow crop rotation."),
    (("redrot", "red_rot"), "Fungal disease (red rot)",
     "No reliable cure in a standing crop. Treat seed setts with Carbendazim or Thiophanate-methyl before planting.",
     "Uproot and destroy infected clumps, apply Trichoderma, grow resistant varieties."),
    (("rust",), "Fungal disease (rust)",
     "Propiconazole, Tebuconazole or Mancozeb.",
     "Remove infected leaves, keep good air circulation in the field."),
    (("mildew",), "Fungal disease (mildew)",
     "Wettable sulphur or Hexaconazole.",
     "Thin dense plants and improve air flow."),
    (("scab", "rot", "mold"), "Fungal disease",
     "Mancozeb, Captan or Chlorothalonil.",
     "Collect and burn fallen infected leaves and fruits."),
    (("blight", "spot", "scorch", "esca"), "Fungal disease (blight / leaf spot)",
     "Mancozeb, Copper oxychloride or Chlorothalonil.",
     "Remove infected leaves, rotate crops, apply Trichoderma or Pseudomonas bio-agents."),
]

def get_medicine(label):
    low = label.lower()
    for keys, kind, med, organic in MEDICINE:
        if any(k in low for k in keys):
            return kind, med, organic
    return "Other", "Please consult your nearest Krishi Vigyan Kendra (KVK) for this disease.", "-"

def action_by_severity(pct, kind):
    if "Viral" in kind:
        if pct >= 10:
            return "Isolate or remove the infected plant and keep the insect vectors under control."
        return "Monitor the crop and keep the insect vectors under control."
    if pct < 10:
        return "Spraying is not urgent yet. Monitor the crop and remove infected leaves. Organic measures may be enough."
    if pct < 30:
        return "Spray the recommended treatment and re-inspect after 10-15 days."
    return "Spray immediately, remove heavily infected leaves or plants, and check again after 7-10 days."

SAFETY = ("Safety: always follow the dose and waiting period printed on the product label, wear gloves and a mask, "
          "and spray in the morning or evening. This is AI-generated guidance; confirm with an agriculture expert or KVK "
          "before treating the crop.")

In [6]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

IMG = 224

def predict(img):
    if img is None:
        return "Please upload an image.", None, {}
    try:
        # Convert any image (PNG, RGBA, grayscale) to RGB and limit its size
        img = img.convert("RGB")
        img.thumbnail((1024, 1024))
        arr = np.array(img)

        x = tf.image.resize(arr, (IMG, IMG))[None, ...]
        p = model.predict(x, verbose=0)[0]
        top = p.argsort()[-3:][::-1]
        probs = {class_names[i].replace("___", " - ").replace("_", " "): float(p[i]) for i in top}
        best = class_names[top[0]]
        conf = float(p[top[0]])
        pct, overlay = estimate_severity(arr)

        if conf < 0.60:
            top2 = ", ".join(f"{k} ({v*100:.0f}%)" for k, v in list(probs.items())[:2])
            return (f"Low confidence. Possible: {top2}. Please upload a clear photo of a single leaf "
                    "on a plain background in good light."), overlay, probs

        crop, _, disease = best.partition("___")
        disease = disease.replace("_", " ")
        if "healthy" in best.lower():
            txt = (f"Crop: {crop}\nCondition: Healthy\nConfidence: {conf*100:.1f}%\n\n"
                   "The leaf looks healthy. No treatment is needed; keep monitoring regularly.")
            return txt, overlay, probs

        kind, med, organic = get_medicine(best)
        if pct is None:
            sev_txt = "Severity: leaf could not be separated from the background (use a plain background)."
            act = "See the treatment advice below."
        else:
            sev_txt = f"Diseased area (severity): {pct:.1f}%  ->  {severity_level(pct)}"
            act = action_by_severity(pct, kind)
        txt = (f"Crop: {crop}\nDisease: {disease}  ({kind})\nConfidence: {conf*100:.1f}%\n{sev_txt}\n\n"
               f"What to do: {act}\n\nRecommended treatment: {med}\nOrganic / cultural measures: {organic}\n\n{SAFETY}")
        return txt, overlay, probs

    except Exception as e:
        return f"Error while processing this image: {e}", None, {}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Leaf image"),
    outputs=[gr.Textbox(label="Result", lines=14),
             gr.Image(label="Diseased area (highlighted in red)"),
             gr.Label(num_top_classes=3, label="Top 3 predictions")],
    title="Crop Leaf Disease Detection, Severity and Treatment (Sugarcane + 14 crops)",
    description="Upload a photo of a sugarcane leaf or any supported crop leaf.",
)
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e21d8a675ee2c89d75.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
